In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.taxi import (
    Route,
    solve_ntinda_equilibrium,
    MovingAverageForecaster,
    ExponentialSmoothingForecaster,
    LinearTrendForecaster,
    SeasonalNaiveForecaster,
    backtest_walk_forward,
    tune_ses_alpha,
    plan_fleet_size,
    simulate_60day_route
)

# Initial Data (10 Days)
routes_data = {
    "Kampala-Ntinda": Route("Kampala-Ntinda", [35, 40, 42, 50, 55, 60, 48, 52, 47, 45], fare_ugx=2000),
    "Kampala-Entebbe": Route("Kampala-Entebbe", [60, 58, 65, 70, 72, 80, 75, 68, 66, 64], fare_ugx=5000),
    "Kampala-Mukono": Route("Kampala-Mukono", [45, 47, 50, 49, 55, 62, 58, 53, 51, 50], fare_ugx=3000)
}

# Route Summary
summary_rows = []
for name, r in routes_data.items():
    stats = r.passenger_stats()
    summary_rows.append({
        "Route": name,
        "Fare (UGX)": f"{r.fare_ugx:,.0f}",
        "Mean Passengers/Day": round(stats["mean"], 1),
        "Total 10-Day Rev (UGX)": f"{r.total_revenue():,.0f}",
        "Std Dev": round(stats["stdev"], 2)
    })

display(pd.DataFrame(summary_rows))

In [ ]:
# Task 2: Solve Ntinda Route Equilibrium
eq_res = solve_ntinda_equilibrium(fare_current=2000.0)

print("--- NTINDA ROUTE EQUILIBRIUM ANALYSIS ---")
print(f"Equilibrium Price (P*):    UGX {eq_res['equilibrium_price_ugx']:,.0f}")
print(f"Equilibrium Quantity (Q*): {eq_res['equilibrium_quantity']:.1f} passengers/trip-hour")
print(f"Current Fare:              UGX {eq_res['current_fare_ugx']:,.0f}")
print(f"Fare Status:               {eq_res['fare_status']}")

Task 2 Equilibrium Interpretation:
The current fare of UGX 2,000 is below the market equilibrium fare of UGX 2,200. At UGX 2,000, market demand ($Q_d = 120 - 0.02(2000) = 80$) exceeds market supply ($Q_s = 10 + 0.03(2000) = 70$), resulting in a shortage of passenger seats during peak hours. Raising the fare towards UGX 2,200 would incentivize operators to supply more vehicle trips while moderating excess passenger demand to achieve market balance. 

In [ ]:
# Task 3 & 4: Model Evaluation via Walk-Forward Backtesting (Days 4-10)
evaluation_rows = []
best_models_per_route = {}

for name, r in routes_data.items():
    passengers = r.daily_passengers
    
    # 1. 3-Day Moving Average
    ma_model = MovingAverageForecaster(window=3)
    mae_ma = backtest_walk_forward(passengers, ma_model, start_day=3)
    
    # 2. Simple Exponential Smoothing (Tuned Alpha)
    best_alpha, mae_ses = tune_ses_alpha(passengers)
    ses_model = ExponentialSmoothingForecaster(alpha=best_alpha)
    
    # 3. Linear Trend
    lt_model = LinearTrendForecaster()
    mae_lt = backtest_walk_forward(passengers, lt_model, start_day=3)
    
    # Select Best Model based on Lowest MAE
    scores = {"Moving Average (3-Day)": mae_ma, f"SES (alpha={best_alpha:.2f})": mae_ses, "Linear Trend": mae_lt}
    best_m_name = min(scores, key=scores.get)
    
    if "Moving Average" in best_m_name:
        best_obj = ma_model
    elif "SES" in best_m_name:
        best_obj = ses_model
    else:
        best_obj = lt_model
        
    best_models_per_route[name] = (best_m_name, best_obj)
    
    evaluation_rows.append({
        "Route": name,
        "MAE (3-Day MA)": round(mae_ma, 2),
        "MAE (SES Tuned)": round(mae_ses, 2),
        "Best Alpha": round(best_alpha, 2),
        "MAE (Linear Trend)": round(mae_lt, 2),
        "Selected Model": best_m_name
    })

df_eval = pd.DataFrame(evaluation_rows)
display(df_eval)

In [ ]:
# Task 5 & 6: Forecast Day 11 Demand, Revenue & Fleet Size
fleet_rows = []

for name, r in routes_data.items():
    best_m_name, best_obj = best_models_per_route[name]
    
    # Predict Day 11 passengers
    day11_pass_pred = best_obj.fit_predict(r.daily_passengers)
    day11_rev_pred = day11_pass_pred * r.fare_ugx
    
    # Plan Fleet Requirements
    fleet_plan = plan_fleet_size(day11_pass_pred, trips_per_vehicle=8, vehicle_capacity=14, buffer_pct=0.15)
    
    fleet_rows.append({
        "Route": name,
        "Best Model": best_m_name,
        "Day 11 Pass Forecast": round(day11_pass_pred, 1),
        "Day 11 Revenue Forecast (UGX)": f"{day11_rev_pred:,.0f}",
        "Raw Vehicles": fleet_plan["raw_vehicles_needed"],
        "Deployed Vehicles (+15% Buffer)": fleet_plan["deployed_vehicles"]
    })

df_fleet = pd.DataFrame(fleet_rows)
display(df_fleet)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

days_hist = np.arange(1, 11)

for idx, (name, r) in enumerate(routes_data.items()):
    ax = axes[idx]
    best_m_name, best_obj = best_models_per_route[name]
    
    hist_rev = r.daily_revenue()
    d11_pred_rev = best_obj.fit_predict(r.daily_passengers) * r.fare_ugx
    
    ax.plot(days_hist, hist_rev / 1000, "bo-", label="Actual Revenue")
    ax.plot(11, d11_pred_rev / 1000, "ro", markersize=8, label=f"Day 11 Forecast ({best_m_name})")
    
    ax.set_title(f"Revenue Forecast: {name}")
    ax.set_xlabel("Day")
    ax.set_ylabel("Revenue ('000 UGX)")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Extension: Simulate 60 Days with Weekly Seasonality
sim_passengers = simulate_60day_route(seed=2026)

ma_3d = MovingAverageForecaster(window=3)
sn_7d = SeasonalNaiveForecaster(season_length=7)

mae_ma_60 = backtest_walk_forward(sim_passengers, ma_3d, start_day=14)
mae_sn_60 = backtest_walk_forward(sim_passengers, sn_7d, start_day=14)

print("\n--- 60-DAY SEASONAL SIMULATION RESULTS ---")
print(f"3-Day Moving Average MAE: {mae_ma_60:.2f} passengers")
print(f"Seasonal-Naïve (7-Day) MAE: {mae_sn_60:.2f} passengers")
print(f"Performance Improvement:    {((mae_ma_60 - mae_sn_60) / mae_ma_60) * 100:.2f}% error reduction")

Findings & Interpretations

Equilibrium analysis for the Ntinda route revealed that the current UGX 2,000 fare is below the UGX 2,200 market equilibrium. This price ceiling creates excess demand and vehicle shortages during rush hours. Adjusting fares toward equilibrium balances commuter demand with operator supply.   In model evaluations, Simple Exponential Smoothing (SES) and Linear Trend models outperformed 3-day moving averages across short 10-day historical horizons. Incorporating a 15% operational buffer into fleet planning ensures routes maintain reliability during sudden demand spikes without over-capitalizing vehicle deployment.  

Extension Insights

When evaluated on a 60-day dataset featuring weekly periodicity (busier Fridays, quieter Sundays), the Seasonal-Naïve model (7-day lag) significantly outperformed the standard moving average. Moving averages introduce lag and smooth out cyclical weekend dips and Friday spikes, whereas seasonal-naïve models preserve day-of-week demand patterns.   

Limitations

1. Unconstrained Demand Assumption: Historical passenger counts reflect observed trips rather than total latent demand, ignoring commuters turned away during peak-hour vehicle shortages.   

2. Static Operational Efficiency: Fleet calculations assume a constant 8 trips/day per vehicle, ignoring traffic congestion variations, driver shift limits, and fuel downtime.